In [0]:
#read raw table
bronze_df = spark.table('workspace.mc311_bronze.service_requests_raw')
#display(bronze_df)

In [0]:
#order dataframe columns and drop
silver_df = bronze_df.select([
    "sr_num",
    "sr_type",
    "sr_area",
    "sr_sub_area",
    "sr_subtype_cd",
    "faq_ques_text",
    "department",
    "sr_stat_id",
    "created",
    "last_upd",
    "closed",
    "number_of_days_open",
    "x_sla",
    "within_sla_window",
    "sla_yes",
    "sla_no",
    "x_city",
    "x_state",
    "x_zipcode",
    "x_councildist",
    "x_councilmbr",
    "x_congdist",
    "x_cong_memb",
    "x_electiondist",
    "x_mdstatedist",
    "ingestion_timestamp_eastern"]
).drop("sla_yes", "sla_no")
display(silver_df)

In [0]:
#rename columns
rename_map = {
    "sr_num": "request_id",
    "sr_type": "request_type",
    "sr_area": "request_area",
    "sr_sub_area": "request_sub_area",
    "sr_subtype_cd": "request_subtype_code",
    "faq_ques_text": "request_description",
    "department": "department",
    "sr_stat_id": "request_status",
    "created": "date_created",
    "last_upd": "date_last_updated",
    "closed": "date_closed",
    "number_of_days_open": "days_open",
    "x_sla": "sla_target",
    "within_sla_window": "within_sla",
    "x_city": "city",
    "x_state": "state",
    "x_zipcode": "zipcode",
    "x_councildist": "council_district",
    "x_councilmbr": "council_member",
    "x_congdist": "congressional_district",
    "x_cong_memb": "congressional_member",
    "x_electiondist": "election_district",
    "x_mdstatedist": "state_district",
    "ingestion_timestamp_eastern": "ingestion_timestamp"}
silver_df = silver_df.toDF(*[rename_map.get(c, c) for c in silver_df.columns])
silver_df.printSchema()

In [0]:
#cast to proper data types
from pyspark.sql.functions import col, when
silver_df = silver_df.withColumns({
    "date_created": col("date_created").cast("timestamp"),
    "date_last_updated": col("date_last_updated").cast("timestamp"),
    "date_closed": col("date_closed").cast("timestamp"),
    "days_open": col("days_open").cast("int"),
    "sla_target": col("sla_target").cast("int"),
    "within_sla": when(col("within_sla") == "Yes", True).when(col("within_sla") == "No", False)
})

silver_df.printSchema()

In [0]:
#check for duplicate rows
from pyspark.sql import functions as F

duplicate_df = (
    silver_df
    .groupBy("request_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
)

display(duplicate_df)

In [0]:
#check nulls in each row
null_counts = [
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in silver_df.columns
]

checked_null_df = silver_df.agg(*null_counts)

display(checked_null_df)
